# DICE — Notebook 1.3 (Uncertainty, Monte Carlo) — Student version

## 0) Setup and import

In [ ]:
# Run this cell once to check/install the Python packages required for this notebook.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["figure.dpi"] = 120
RNG = np.random.default_rng(42)

def clone_params(base, **overrides):
    candidate = Params()
    for name, value in vars(base).items():
        setattr(candidate, name, value)
    for name, value in overrides.items():
        setattr(candidate, name, float(value))
    return candidate

def simulate(base, **overrides):
    candidate = clone_params(base, **overrides)
    path = init_states(candidate)
    path[1:, candidate.i_s] = 0.20
    path[1:, candidate.i_mu] = np.linspace(0.03, 0.60, candidate.nT - 1)
    path = update_path(path, range(1, candidate.nT), candidate)
    return path, candidate

def damage_fraction(path, par):
    lagged_temperature = np.r_[path[0, par.i_T_AT], path[:-1, par.i_T_AT]]
    damages = par.a2 * lagged_temperature ** par.a3
    if par.a4 != 0:
        damages += np.where(lagged_temperature > par.a6,
                            par.a4 * lagged_temperature ** par.a5, 0.0)
    return damages

def quantile_bands(array):
    return np.quantile(np.asarray(array), [0.05, 0.50, 0.95], axis=0)


## Q1) Simulate the baseline simulation (one scenario).

Let us consider a baseline scenario. Simulate the DICE model with baseline calibration considering for controls (exogenous): constant saving `s_t = 0.20`; abatement `mu_t` ramps 0.03 -> 0.60 in 2060. Report path (`T_AT`) and  (`Y`) .

In [ ]:
# simulate(...) fixes the saving and abatement paths described above.
p = Params()
baseline, _ = simulate(p)
years = baseline[:, p.i_time]

# Plot baseline[:, p.i_T_AT] and baseline[:, p.i_Y] in two panels.


## Q2) Consider now parametric uncertainty about temperature sensitivity T2XCO2.

In Nordhaus (2018), one can read that:  
> *“Equilibrium temperature sensitivity (ETS). The distribution for ETS adopts the approach used in the MUP study. The primary estimates are from Olsen et al. (2012). This study uses a Bayesian approach, with a prior based on previous studies and a likelihood based on observational or modeled data. The best fitting distribution is a log-normal PDF. The parameters of the log-normal distribution fit to Olsen et al. are μ = 1.107 and σ = 0.264. The major summary statistics of the reference distribution in the study are the following: mean = 3.13, median = 3.03 and standard deviation = 0.843.”*


### Q2A) Draw 1,000 observations of `T2XCO2` and report the histogram.

*Hints:*
- Use a lognormal distribution centred on the baseline `p.T2XCO2`.
- Set `sigma_ecs = math.log(1.25) / norm.ppf(0.95)`.
- Draw with `np.exp(math.log(p.T2XCO2) + sigma_ecs * RNG.standard_normal(N))`.
- Plot the histogram and report the 5%, 50% and 95% quantiles.


In [ ]:
N = 1000
sigma_ecs = math.log(1.25) / norm.ppf(0.95)
# ecs_draws = ...


### Q2B) Create a loop that generates the update path and stores them.
*Hints:*
- Start with `p = Params()`, `sim = init_states(p)`.  
- For each draw, create a new parameter object and override: `p2.T2XCO2 = float(t2x)`.  
- Set controls as in baseline (constant `s`, linear ramp for `mu`).  
- Run `update_path(sim2, range(1, p2.nT), p2)`.  
- Store results in lists, e.g. `T_list.append(sim2[:, p2.i_T_AT])`, `Y_list.append(sim2[:, p2.i_Y])`, and damages `D_list`  (compute it manually).


In [ ]:
# Your code for question 2-B should be here


### Q2C) Report the confidence interval for temperatures, GDP and damages.
*Hints:*
- Use `np.quantile(arr, [0.05,0.5,0.95], axis=0)` to compute 5–50–95% bands.  
- Plot fan charts with `plt.fill_between(years, q05, q95, alpha=0.2)` and `plt.plot(years, q50)`.  
- Extract values in 2100 with `i2100 = np.argmin(np.abs(years-2100))` and print `q05[i2100], q50[i2100], q95[i2100]`.  


In [ ]:
# Your code for question 2-C should be here

> ✍️ You written answer here.

## Q3) Consider now parametric uncertainty about the damage parameter a2.

In Nordhaus (2018), one can read that:  
> *“Given the different approaches, I settled on a value for the uncertainty of the damage parameter which is one-half the mean value of the parameter. More precisely, the distribution is assumed to be normal, with a standard deviation of 0.118% Y/°C². This reflects the great divergence today among different studies.”*



### Q3A) Draw 1,000 observations of the damage parameter `a2` and report the histogram.

*Hints:*
- Use a positive lognormal distribution centred on `p.a2`.
- Set `sigma_a2 = math.log(2.0) / norm.ppf(0.95)`.
- Draw with `np.exp(math.log(p.a2) + sigma_a2 * RNG.standard_normal(N))`.
- Plot the histogram and report the 5%, 50% and 95% quantiles.


In [ ]:
# Your code for question 3 should be here

### Q3B) Simulate joint uncertainty in climate sensitivity and damages.

*Hints:*
- Loop over `zip(ecs_draws, a2_draws)`.
- For each pair, call `simulate(p, T2XCO2=ecs, a2=a2)`.
- Store temperature, output and `damage_fraction(path, par)`.
- Convert the three lists to NumPy arrays before computing quantiles.


In [ ]:
# Your code for question 3 should be here

### Q3C) Report the confidence interval for temperatures, GDP and damages.  Interpret.
*Hints:*
- Use `np.quantile(arr, [0.05,0.5,0.95], axis=0)` across the simulated ensemble.  
- Plot fan charts with `plt.fill_between(years, q05, q95, alpha=0.2)` and `plt.plot(years, q50)`.  
- Report values for 2100: find index with `i2100 = np.argmin(np.abs(years-2100))` and print the quantiles for `T_AT`, `Y`, and damages.  

In [ ]:
# Your code for question 3 should be here

> ✍️ You written answer here.

### Q3D) Compare ECS-only uncertainty with joint ECS + damage uncertainty.

Plot the two 5–95% temperature bands on the same axes. Then compare their
widths in 2100 using `q95 - q05` for temperature, output and damages.


In [ ]:
# Your code for question 3 should be here

> ✍️ You written answer here.

## Q4) Consider now parametric uncertainty about the decarbonization parameter σ(t).

In Nordhaus (2018), one can read that:  
> *“The simplest approach is to estimate an OLS regression, using data from 1960 to 2015, and then look at the forecast error for 2100. If an AR1 term is included in the equation, the standard error of the forecast for 2100 is 13.5% of the logarithm of σ(t). This implies an annual uncertainty of 0.149% per year. However, a unit root of σ(t) cannot be rejected, so this estimate is biased downward.”*


### Q4A) Draw 1,000 observations of the decarbonisation-trend parameter.

The default `deltasig` is zero. For this transparent stress exercise, use a
positive half-normal distribution:

```python
deltasig_draws = np.abs(RNG.normal(loc=0.0, scale=0.02, size=N))
```

Plot the histogram and report the 5%, 50% and 95% quantiles.


In [ ]:
# Your code for question 4 should be here

### Q4B) Simulate all three sources of uncertainty jointly.

*Hints:*
- Loop over `zip(ecs_draws, a2_draws, deltasig_draws)`.
- Call `simulate(p, T2XCO2=ecs, a2=a2, deltasig=deltasig)`.
- Store temperature, output and damages exactly as in Q3B.


In [ ]:
# Your code for question 4 should be here

### Q4C) Report the confidence interval for temperatures, GDP and damages.
*Hints:*
- Use `np.quantile(arr, [0.05,0.5,0.95], axis=0)` to compute confidence bands across the ensemble.  
- Plot fan charts with `plt.fill_between(years, q05, q95, alpha=0.2)` and overlay the median with `plt.plot(years, q50)`.  
- Extract 2100 values with `i2100 = np.argmin(np.abs(years-2100))` and print the 5–50–95% range.  

In [ ]:
# Your code for question 4 should be here

In [ ]:
# Intentionally left as a workspace cell.